# 03 — Custom CNN: Training & Evaluation

This notebook trains the custom CNN architecture and evaluates it on the held-out test set.

**Expected result:** ~97% test accuracy

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger

from models import build_custom_cnn
from preprocessing import load_splits, get_data_generators
from evaluate import plot_training_curves, evaluate_model

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 1. Load Data

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = load_splits('../data')

print(f'Train : {X_train.shape}  labels: {y_train.shape}')
print(f'Val   : {X_val.shape}  labels: {y_val.shape}')
print(f'Test  : {X_test.shape}  labels: {y_test.shape}')

train_gen, val_gen = get_data_generators(X_train, y_train, X_val, y_val)

## 2. Build Model

In [ ]:
model = build_custom_cnn()

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

model.summary()

## 3. Train

In [ ]:
EPOCHS     = 50
BATCH_SIZE = 32

callbacks = [
    ModelCheckpoint('../results/saved_models/cnn_best.h5',
                    monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=10,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=5, min_lr=1e-6, verbose=1),
    CSVLogger('../results/training_logs/cnn_training_log.csv')
]

history = model.fit(
    train_gen,
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    validation_data=val_gen,
    validation_steps=len(X_val) // BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

## 4. Training Curves

In [ ]:
plot_training_curves(history, model_name='cnn')

## 5. Test Set Evaluation

In [ ]:
metrics = evaluate_model(model, X_test, y_test, model_name='cnn')
print('\nFinal Metrics:')
for k, v in metrics.items():
    if k != 'model':
        print(f'  {k:12s}: {v:.4f}')